# 🔮 05 — LSTM Model (baseline)
Train a simple LSTM on the final dataset. This notebook uses the `data/processed/final_dataset.csv` produced by `src/preprocessing/build_dataset.py`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

DATA = Path('..') / 'data' / 'processed' / 'final_dataset.csv'
if not DATA.exists():
    print('final_dataset.csv not found. Please run src/preprocessing/build_dataset.py or create the file.')
df = pd.read_csv(DATA) if DATA.exists() else pd.DataFrame()
df.shape

## Prepare a simple univariate sequence for `market_value_eur` (toy example)
If your dataset has multiple players/timepoints, reshape accordingly. For the sample data we will create a tiny sequence per player by using the single value (works as demo).

In [ ]:
if df.empty:
    print('No data to train on. Exiting this notebook.')
else:
    # For demo: use features as X and target as market_value_eur
    X = df.select_dtypes(include=[np.number]).drop(columns=['market_value_eur'], errors='ignore').fillna(0)
    y = df['market_value_eur'] if 'market_value_eur' in df.columns else np.zeros(len(df))
    if X.shape[0] < 2:
        print('Not enough rows to train LSTM in demo mode.')
    else:
        scaler = MinMaxScaler()
        Xs = scaler.fit_transform(X)
        # reshape to (samples, timesteps=1, features)
        Xs = Xs.reshape((Xs.shape[0], 1, Xs.shape[1]))
        X_train, X_test, y_train, y_test = train_test_split(Xs, y, test_size=0.2, random_state=42)
        model = Sequential([
            LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mse')
        print(model.summary())
        history = model.fit(X_train, y_train, epochs=10, batch_size=8, validation_data=(X_test, y_test))
        preds = model.predict(X_test)
        print('Preds shape:', preds.shape)
